In [1]:
import pandas as pd
import joblib
import importlib

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
import src.models.model_trainer as mt

importlib.reload(mt)

<module 'src.models.model_trainer' from 'd:\\DIVY\\CODING\\PURECODING\\AI-ML\\Week9\\Devops\\JobClarity\\src\\models\\model_trainer.py'>

In [4]:
vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

In [6]:
df = pd.read_csv(
    "../data/processed/cleaned_jobs.csv",
    keep_default_na=False
)

In [7]:
import src.features.feature_builder as fb

importlib.reload(fb)

df = fb.create_combined_text(df)

X = vectorizer.transform(df["combined_text"])

y = df["fraudulent"]

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
model = mt.create_model()

print(model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)


In [10]:
model = mt.train_model(
    model,
    X_train,
    y_train
)

print("✅ Model Training Completed")

✅ Model Training Completed


In [11]:
print(model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)


In [12]:
y_pred = model.predict(X_test)

print(y_pred[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [13]:
y_prob = model.predict_proba(X_test)

print(y_prob[:5])

[[9.7758681e-01 2.2413192e-02]
 [9.9913394e-01 8.6607173e-04]
 [9.9829763e-01 1.7023627e-03]
 [9.9672186e-01 3.2781651e-03]
 [9.9892485e-01 1.0751667e-03]]


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [15]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob[:,1])

In [16]:
print(f"Accuracy : {accuracy:.4f}")

print(f"Precision: {precision:.4f}")

print(f"Recall   : {recall:.4f}")

print(f"F1 Score : {f1:.4f}")

print(f"ROC AUC  : {roc_auc:.4f}")

Accuracy : 0.9815
Precision: 0.9735
Recall   : 0.6358
F1 Score : 0.7692
ROC AUC  : 0.9887


In [17]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       0.97      0.64      0.77       173

    accuracy                           0.98      3576
   macro avg       0.98      0.82      0.88      3576
weighted avg       0.98      0.98      0.98      3576



In [18]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[3400    3]
 [  63  110]]


In [19]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(exist_ok=True)

In [20]:
joblib.dump(
    model,
    MODEL_DIR / "xgboost_model.pkl"
)

print("✅ Model Saved Successfully")

✅ Model Saved Successfully


In [21]:
loaded_model = joblib.load(
    MODEL_DIR / "xgboost_model.pkl"
)

print(type(loaded_model))

<class 'xgboost.sklearn.XGBClassifier'>


In [22]:
loaded_pred = loaded_model.predict(X_test)

print(loaded_pred[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [27]:
import importlib
import src.models.model_loader as loader

importlib.reload(loader)

model = loader.load_model()
vectorizer = loader.load_vectorizer()

print(type(model))
print(type(vectorizer))

<class 'xgboost.sklearn.XGBClassifier'>
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [28]:
from pathlib import Path

print(Path("../models/xgboost_model.pkl").resolve())
print(Path("../models/xgboost_model.pkl").exists())

D:\DIVY\CODING\PURECODING\AI-ML\Week9\Devops\JobClarity\models\xgboost_model.pkl
True


True
